## 1. Initialize the project

Fetch the working context: import the project 'rag-text-serve'. Project is a placeholder for the code, data, and management of the data operations inside the digitalhub platform.

In [ ]:
import digitalhub as dh
PROJECT_NAME = "rag-text-serve"
proj = dh.get_or_create_project(PROJECT_NAME)

## 2. Execution

### 2.1 Deploy VLLM serve
Fetch the "vllm-text-serve" operation in the project. It deploys a large language model (LLM) using the VLLM engine within the digitalhub platform. For more detailed information, please refer to the catalog function [LLM VLLM Serve](https://scc-digitalhub.github.io/hub/functions/llm-vllm-text-serve)

In [ ]:
llm_vllm_func = proj.get_function("vllm-text-serve") 

 Run the function with a GPU profile. For more details about how to create, run, and view jobs inside to the digital hub platform see the <a href ="https://scc-digitalhub.github.io/docs/0.14/runtimes/python/">documentation </a>

In [ ]:
vllm_run = llm_vllm_func.run("serve", profile="1xV100", use_cpu_image=False, wait=True)

Wait for job to finish. The state of 'serve' run can be viewed running the cell below.



In [ ]:
service = vllm_run.refresh().status.service

 Now fetch the CHAT_URL and CHAT_MODEL variables for communicating with the deployed LLM service

In [ ]:
CHAT_URL = "http://"+vllm_run.status.to_dict()["service"]["url"]
CHAT_MODEL = vllm_run.status.to_dict()["openai"]["model"]
print(f"service {CHAT_URL} with model {CHAT_MODEL}")

#### Test VLLM Serve

Send a test request to the deployed VLLM service to verify it is running correctly. We will use the OpenAI-compatible API endpoint to send a chat completion request with the deployed model.

In [ ]:
json_payload = {'model': CHAT_MODEL, 'prompt': 'Describe MLOPs'}
json_payload

In [ ]:
import pprint
pp = pprint.PrettyPrinter(indent=2)
result = vllm_run.invoke(json=json_payload, url='http://' + service['url']+'/v1/completions').json()
print("Response:")
pp.pprint(result)

### 2.2 Embedding
We will now fetch a function 'embed' from the project which will convert text documents into vector embeddings for semantic search and retrieval.


In [ ]:
embed_function = proj.get_function("embed")

The 'embed' function is a kubeai-text function that converts text documents into vector embeddings for semantic search and retrieval. The function processes the extracted text content and generates dense vector representations that capture semantic meaning, enabling efficient similarity-based searches across the document corpus. 
We deploy the function as a VLLM serve instance to handle embedding requests efficiently at scale.

In [ ]:
embed_run = embed_function.run("serve", wait=True)

Wait until the `embed` service is fully deployed, then refresh the run status and verify it is in a ready/running state before continuing.

In [ ]:
status = embed_run.refresh().status
print("Service status:", status.state)

Fetch the `EMBED_URL` and `EMBED_MODEL` variables from the embed service status to establish connection parameters for sending embedding requests to the deployed service.


In [ ]:
EMBED_URL = status.to_dict()["service"]["url"]
EMBED_MODEL = status.to_dict()["openai"]["model"]
print(f"service {EMBED_URL} with model {EMBED_MODEL}")


### 2.3 Text Extraction

Before proceeding to the generate embedding step, it is important to extract and generate an artifact from the source document in PDF document. For more detailed information, please refer to the catalog function [Text Extraction Service](https://scc-digitalhub.github.io/hub/functions/text-extraction-service) that demonstrates how to convert a PDF document into text format, which can then be used for generating embeddings. You are free to change the address or local path to any PDF file you prefer. For the purposes of this tutorial, the function demonstrates processing a PDF file downloaded from a local PDF folder 'documents'.


In [ ]:
html_artifact = proj.get_artifact("document.pdf_output.html")

### 2.4 Embedder Function

Fetch the 'embedder' function from the project. This function is a Python function that will process the extracted text content from the PDF and generate vector embeddings that capture semantic meaning, enabling efficient similarity-based searches across the document corpus. The embedder function utilizes the embedding service deployed earlier to convert text into dense vector representations.

In [ ]:
embedder_function = proj.get_function("embedder")

Execute the 'embedder' function to generate vector embeddings from the extracted HTML content. The function orchestrates the embedding generation process by:

1. **Input**: Takes the HTML artifact from the text extraction step as input
2. **Parameters**: Uses environment variables to configure the embedding service deployed in the previous step:
    - `EMBEDDING_SERVICE_URL`: The URL of the deployed embedding service in step 2.2
    - `EMBEDDING_MODEL_NAME`: The model name used by the embedding service in step 2.2
3. **Processing**: Converts the extracted text into dense vector representations that capture semantic meaning through the embedding service
4. **Output**: Stores the generated embeddings in the database for efficient similarity-based searches across the document corpus

The function coordinates the entire embedding pipeline, from text processing through the embedding service to persistent storage of the vectorized representations.

embedder_run = embedder_function.run(
    "job",
    inputs={"input": html_artifact.key},
    envs=[
        {
            "name": "EMBEDDING_SERVICE_URL",
            "value": EMBED_URL
        },
        {    "name": "EMBEDDING_MODEL_NAME",
            "value": EMBED_MODEL,
        }
    ],
    wait=True,
)

### 2.5 RAG Service

Fetch the 'rag-service' function from the project. This function implements a Retrieval-Augmented Generation (RAG) pipeline that combines the deployed embedding and LLM services to enable intelligent document-based question answering. The RAG service leverages the vector embeddings generated in the previous step to retrieve semantically similar documents from the corpus, and then uses the deployed LLM to generate contextually accurate responses based on the retrieved documents.


In [ ]:
serve_func = proj.get_function("rag-service")

Run the 'rag-service' function as a service with the required resources and environment variables. This deploys the RAG pipeline that integrates the embedding and LLM services for question answering:

- **Resources**: Allocates 4 CPU cores and 4Gi of memory for the service
- **Environment Variables**: Configures the service with:
    - `CHAT_MODEL_NAME`: The LLM model deployed in step 2.1
    - `CHAT_SERVICE_URL`: The URL of the deployed VLLM service
    - `EMBEDDING_MODEL_NAME`: The embedding model deployed in step 2.2
    - `EMBEDDING_SERVICE_URL`: The URL of the deployed embedding service

The RAG service uses these components to retrieve semantically relevant documents based on user queries and generate contextually accurate responses using the deployed LLM.

In [ ]:
serve_run = serve_func.run(
    action="serve",
    resources={
        "cpu": "4",
        "mem": "4Gi",
    },
    envs=[
            {"name": "CHAT_MODEL_NAME", "value": CHAT_MODEL},
            {"name": "CHAT_SERVICE_URL", "value": CHAT_URL},
            {"name": "EMBEDDING_MODEL_NAME", "value": EMBED_MODEL},
            {"name": "EMBEDDING_SERVICE_URL", "value": EMBED_URL}
         ],
    wait=True
)

Once the 'rag-service' is deployed, fetch the `AGENT_URL` from the service status. This URL is the endpoint for accessing the deployed RAG service to send queries and receive contextually accurate responses based on the document corpus.

In [ ]:
AGENT_URL = serve_run.status.to_dict()["service"]["url"]
print(AGENT_URL)

#### Query the RAG Service

Now that the RAG service is deployed, you can send queries to retrieve answers based on the embedded document corpus. The service will:

1. **Embed the Query**: Convert your question into vector embeddings using the deployed embedding service
2. **Retrieve Documents**: Search the document corpus to find semantically similar content
3. **Generate Response**: Use the deployed LLM to generate contextually accurate answers based on retrieved documents

Send a POST request to the RAG service endpoint with your question. The service will return answers grounded in the document corpus that was processed in the text extraction and embedding generation steps.
```

In [ ]:
import requests

res = requests.post(f"http://{AGENT_URL}",json={"question": "what is PAT token"})
print(res.json())